# Data Fix and Construct

While trying to give some result fast, data acquisition is going to be poor day by day.
Therefore, i wanna create a notebook to fix this issue and create a bigger DataFrame which has all.

In [1]:
import pandas as pd
import numpy as np
from preprocessing import *


In [212]:
# some helper functions

def get_row_by_name(df,name):
    return df[df.name == name]
def get_row_by_date(df,date):
    return df[df.date == date]
    
def get_row_by_name_date(df,name,date):
    return get_row_by_date(get_row_by_name(df,name),date)

# note: supports only one etf at a time
def get_row_by_min_max_date(df, min_date, max_date):
        
    row_min = get_row_by_date(df,min_date)
    row_max = get_row_by_date(df,max_date)
    if len(row_min) == 0: # empty df
        raise Exception("min index not found!")
    if len(row_max) == 0: # empty df
        raise Exception("max index not found!")
    
    idx_min = row_min.index[0]
    idx_max = row_max.index[0]
    
    return df.iloc[idx_min:idx_max+1]

In [213]:
#this function gives proper column names to the image with label dataframe.
def fix_image_w_label_df(image_w_label_df):
    col_names = []
    col_names.append('name')
    col_names.append('date')
    for i in range(28):
        for j in range(28):
            col_names.append('px_{}_{}'.format(i,j))
    col_names.append('im_label')
    image_w_label_df.columns = col_names
    image_w_label_df.drop('name',axis=1, inplace=True)
    return image_w_label_df

In [231]:
# this function is the main.
# 1. it calls all csv files inside given paths 
# 2. fix image labels
# 3. merge dataframes into one big
# 4. reorder column names for visualization
def call_all_csv(s_w_label_path="../input/stock_with_labels",i_w_label_path="../input/images_with_labels" ):
    import os
    final_df = pd.DataFrame()
    
    for file in os.listdir(s_w_label_path):
        stock_w_label = pd.read_csv(s_w_label_path+"/{}".format(file)) # call stock with label csvs
        image_w_label = pd.read_csv(i_w_label_path+"/{}".format(file), header=None) # call image with label csvs
        image_w_label = fix_image_w_label_df(image_w_label) # fix image with label csv
        df = pd.merge(left=stock_w_label, right=image_w_label, on='date') # merge stock and image data wrt date column
        df['name'] = [file.split('.')[0]]*df.shape[0] # create name column to save stock name
        
        final_df = pd.concat((final_df,df)) # concatenate with previously big dataframe
    
    # lets reorder columns
    # Reordering gives clear visualization when i call df.head()
    col_names = final_df.columns
    ordered_col_names = ['name','date','adjusted_close','pct_change_tanh', # important features
                         'label_day_tanh_regr', 'label_day_is_less','label_day_is_more','im_label', #labels
                         'volume' ,'open','high','low','close'] # not that important features
    
    for col in final_df.columns.values:
        if not col in ordered_col_names:
            ordered_col_names.append(col)
            
    return final_df[ordered_col_names]

In [232]:
final_df = call_all_csv()

In [233]:
final_df.head()

,name,date,adjusted_close,pct_change_tanh,label_day_tanh_regr,label_day_is_less,label_day_is_more,im_label,volume,open,...,px_27_18,px_27_19,px_27_20,px_27_21,px_27_22,px_27_23,px_27_24,px_27_25,px_27_26,px_27_27
0,dia,2000-04-06,111.42,0.736231,-0.428230,1.0,0.0,-0.428230,2279700.0,111.38,...,0.824468,0.969329,0.165875,0.134503,0.096401,0.066682,1.647940,0.690097,2.132665,2.150274
1,dia,2000-04-07,110.91,-0.428230,0.776652,0.0,1.0,0.776652,1027100.0,111.97,...,0.635608,1.023835,0.134960,0.207121,0.191436,-0.866163,1.820766,1.176481,1.994617,2.201748
2,dia,2000-04-10,112.06,0.776652,0.786402,0.0,1.0,0.786402,1833500.0,111.22,...,0.584117,0.969150,-0.379005,0.170153,-0.121516,-0.284304,1.996653,1.644706,1.883828,2.212428
3,dia,2000-04-11,113.25,0.786402,-0.984014,1.0,0.0,-0.984014,3624800.0,112.25,...,0.483722,1.017058,-0.938307,-0.241984,-0.405416,0.998436,2.182907,2.010879,1.770700,2.219715
4,dia,2000-04-12,110.52,-0.984014,-0.873621,1.0,0.0,-0.873621,3620700.0,113.88,...,-0.114792,0.738233,1.411892,1.750000,1.004494,0.930762,2.210079,2.178750,1.641094,2.108255


In [ ]:
df.to_pickle("../input/fixed_data.pickle") # save huge data

In [ ]:
df = pd.read_pickle("../input/fixed_data.pickle") # this is how we read dı dım dıt dıdım:)

# # Some example usages

In [235]:
# lets pull test data
data = get_last_saved_data("../input/last_saved_data")

first_date = data['test_dates'].values[:,0][0] #get first date in data
last_date = data['test_dates'].values[:,0][-1] #get last date in data
data.keys()

dict_keys(['test_dates', 'train_names', 'test_names', 'train_images', 'test_labels', 'train_labels', 'train_dates', 'test_images'])

In [254]:
# usage of 'get_row_by_name'
spy_df = get_row_by_name(final_df,'spy')

In [255]:
# usage of 'get_row_by_date'
first_date_df = get_row_by_date(final_df,first_date)

In [261]:
# usage of 'get_row_by_name_date'
first_spy_df = get_row_by_name_date(final_df, 'spy', first_date)
first_spy_df

,name,date,adjusted_close,pct_change_tanh,label_day_tanh_regr,label_day_is_less,label_day_is_more,im_label,volume,open,...,px_27_18,px_27_19,px_27_20,px_27_21,px_27_22,px_27_23,px_27_24,px_27_25,px_27_26,px_27_27
3793,spy,2015-03-24,208.82,-0.509389,-0.898692,1.0,0.0,-0.898692,77805321.0,209.85,...,-0.420417,-0.554152,1.063933,1.419425,-0.073105,0.569507,-0.789397,0.863321,0.948554,1.010242


In [256]:
from profit_loss import *

p_l = Profit_Loss(capital=100000)
s = p_l.construct_buy_hold_sell_signal(['spy'], method="custom", prediction=predictions_test)
t,profit = p_l.transaction(prices_test, s)
plt.scatter(list(range(len(s))),s)
plt.show()
print(profit)

3791

In [257]:
spy_df.iloc[3791]



name                             spy
date                      2015-03-20
adjusted_close                210.41
pct_change_tanh             0.408965
label_day_tanh_regr        -0.192428
label_day_is_less                  1
label_day_is_more                  0
im_label                   -0.192428
volume                   1.77715e+08
open                          209.71
high                          211.02
low                           209.49
close                         210.41
rsi_15                       55.8242
rsi_20                       55.2326
rsi_25                       54.8928
rsi_30                       54.6717
sma_15                       208.599
sma_20                       209.284
sma_25                       209.476
sma_30                       208.992
macd_26_12                   1.28497
macd_28_14                   1.30946
macd_30_16                   1.31917
macd_trigger_9_26_12       -0.174702
macd_trigger_10_28_14      -0.123547
macd_trigger_11_30_16     -0.0812119
w